In [ ]:
# jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from isaacsim.examples.interactive.base_sample import BaseSample
from isaacsim.core.api import World
from isaacsim.core.utils.nucleus import get_assets_root_path
from isaacsim.core.utils.types import ArticulationAction
from isaacsim.core.api.robots import Robot

import isaacsim.core.utils.stage as stage_utils

import numpy as np

In [ ]:
class HelloWorld(BaseSample):
    def __init__(self) -> None:
        super().__init__()

        # define assets root path
        self._isaac_assets_path = get_assets_root_path()
        
        # define url of assets
        self.jetbot_url = self._isaac_assets_path + "/Isaac/Robots/NVIDIA/Jetbot/jetbot.usd"
        
        return

    def setup_scene(self):
        world = self.get_world()
        world.scene.add_default_ground_plane() # https://docs.isaacsim.omniverse.nvidia.com/5.0.0/py/source/extensions/isaacsim.core.api/docs/index.html#isaacsim.core.api.world.World
        stage_utils.add_reference_to_stage(usd_path=self.jetbot_url, prim_path="/World/jetbot")
        # kaya robot의 기능을 사용하기 위해 wrapper class 사용
        jetbot = world.scene.add(Robot(prim_path="/World/jetbot", name="jetbot1"))     
        return
    
    # right after the scene setup 
    async def setup_post_load(self):
        self._world = self.get_world()
        self._jetbot = self._world.scene.get_object("jetbot1") # load by name
        # print(self._jetbot.__dir__()) 
        # print(self._jetbot.dof_names)
        self._jetbot_articulation_controller = self._jetbot.get_articulation_controller()
        self._world.add_physics_callback("apply_actions", callback_fn=self.apply_robot_actions)


    def apply_robot_actions(self, step_size):
         self._jetbot_articulation_controller.apply_action(ArticulationAction(
                                                                             # joint_positions=None,
                                                                              joint_efforts=None,
                                                                              joint_velocities= np.array([3,3]),
                                                                              # joint_velocities= 5 * np.random.rand(2,)
                                                                             ))


In [ ]:
await HelloWorld().load_world_async()